# 🖼️ Whose Dime — images and clips

**The pieces of one video by the Gemini API, written straight into its Drive folder** (session S11): `images/scene_NNN.png` (2752x1536) for every `scene_NNN` prompt of `03_scenes.md`, then `clips/scene_NNN.mp4` (8 s, 1080p) for every `anim_NNN` prompt, from its image. All the logic is in `whosedime/generate/` (`images.py`, `clips.py`, `run.py`), tested with a fake API; every number (models, prices with their date, sizes) in `whosedime/generate/settings.json`.

**Money is Franco's (D18).** `estimate` spends nothing; `test` makes 3 images and 1 clip at most; `full` stops before passing `APPROVED_USD`, counting what the ledger already holds. An image or clip already in the folder is kept (run again after a failure: it resumes). To redo one, delete its file and run again.

In [ ]:
# ── Cell 1: settings ─────────────────────────────────────────────────────────
VIDEO = ""  # @param {type:"string"}
# The video folder name, e.g. "01_uber" (it must hold 03_scenes.md).

MODE = "estimate"  # @param ["estimate", "test", "full"]
# estimate: the price of what is missing, no call · test: 3 images and 1 clip at most · full: every missing image and clip

APPROVED_USD = 0.0  # @param {type:"number"}
# With MODE = "full": the amount you agree to spend on this video (decision D18). The run stops before passing it.

ROOT = "/content/drive/MyDrive/WhoseDime"  # @param {type:"string"}
# The WhoseDime folder in your Drive.


In [ ]:
# ── Cell 2: code, Drive and key ─────────────────────────────────────────────
import os, subprocess, sys
if not globals().get("REPO_READY"):  # opened directly, not through the launcher: take main (D5)
    subprocess.run(["rm", "-rf", "/content/Channel"], check=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                       "https://github.com/Johnfrancis6/Channel.git", "/content/Channel"]).returncode:
        raise SystemExit("❌ Could not fetch the code (branch main).")
sys.path.insert(0, "/content/Channel/whosedime/notebooks")
sys.path.insert(0, "/content/Channel/whosedime/generate")
for name in ("gemini", "images", "clips", "run", "checker", "launcher"):
    sys.modules.pop(name, None)
import launcher
COMMIT = globals().get("COMMIT") or launcher.commit()
print(f"✓ Code: main @ {COMMIT}")

from pathlib import Path
ROOT = Path(ROOT)
if not ROOT.is_dir() and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not ROOT.is_dir():
    raise SystemExit(f"❌ No WhoseDime folder at {ROOT}. Fix ROOT in cell 1.")
if MODE != "estimate" and not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
    except Exception as e:  # secret missing or not allowed: say so, never print a key
        raise SystemExit(f"❌ No GEMINI_API_KEY secret ({type(e).__name__}): add it in 🔑 Secrets and allow this notebook.")


In [ ]:
# ── Cell 3: the video folder ────────────────────────────────────────────────
FOLDER = ROOT / "Videos" / VIDEO.strip()
if not VIDEO.strip() or not (FOLDER / "03_scenes.md").is_file():
    have = sorted(p.name for p in (ROOT / "Videos").glob("*/03_scenes.md"))
    raise SystemExit(f"❌ VIDEO must name a folder with 03_scenes.md. With one: {[Path(h).parent.name for h in have] or 'none yet'}")
if MODE not in ("estimate", "test", "full"):
    raise SystemExit(f"❌ MODE '{MODE}' is not estimate, test or full")
if MODE == "full" and not APPROVED_USD:
    raise SystemExit("❌ MODE = 'full' needs APPROVED_USD: run 'estimate' first and agree an amount (D18).")
print(f"🖼️ Video: {FOLDER.name}  ·  mode {MODE}" + (f" (up to ${APPROVED_USD:.2f})" if MODE == "full" else ""))


In [ ]:
# ── Cell 4: estimate, or generate, then the status ──────────────────────────
import images, clips, run as runner
images.main([str(FOLDER), "--estimate"])
clips.main([str(FOLDER), "--estimate"])
if MODE != "estimate":
    args = type("Args", (), dict(test=(MODE == "test"), approved_usd=float(APPROVED_USD or 0),
                                 no_render=True, work=None))
    STATUS = runner.run(FOLDER, args)
    print(f"\n💵 Spent on this video so far: ${STATUS['spent_usd']:.2f} ({STATUS['paid_calls']} paid calls)")
    print("Next:", STATUS["next"])


In [ ]:
# ── Cell 5: look at the newest images ───────────────────────────────────────
try:
    from IPython.display import Image as Show, display
except ImportError:
    display = None
if MODE != "estimate" and display:
    for img in sorted((FOLDER / "images").glob("scene_*.png"), key=lambda p: p.stat().st_mtime)[-6:]:
        print(img.name)
        display(Show(str(img), width=480))
